In [1]:
import pandas as pd
import numpy as np
from src.config.path import RAW_PATH, PROCESS_PATH

In [2]:
OUT_PUT = PROCESS_PATH / "flight.parquet"
 
df = pd.read_parquet(RAW_PATH / "drone.parquet", engine="pyarrow")
df

,flight,time,wind_speed,wind_angle,battery_voltage,battery_current,position_x,position_y,position_z,orientation_x,...,angular_z,linear_acceleration_x,linear_acceleration_y,linear_acceleration_z,speed,payload,altitude,date,time_day,route
0,1,0.00,0.1,12.0,24.222174,0.087470,-79.782396,40.458047,269.332402,0.001772,...,0.006815,0.004258,-0.120405,-9.811137,4.0,0.0,25,2019-04-07,10:13,R5
1,1,0.20,0.1,3.0,24.227180,0.095421,-79.782396,40.458047,269.332056,0.001768,...,0.002034,0.006175,-0.116397,-9.810392,4.0,0.0,25,2019-04-07,10:13,R5
2,1,0.30,0.1,352.0,24.225929,0.095421,-79.782396,40.458047,269.333081,0.001768,...,-0.000874,0.002696,-0.128592,-9.809440,4.0,0.0,25,2019-04-07,10:13,R5
3,1,0.50,0.1,354.0,24.224678,0.095421,-79.782396,40.458047,269.334648,0.001775,...,0.002443,0.002024,-0.128271,-9.810159,4.0,0.0,25,2019-04-07,10:13,R5
4,1,0.60,0.1,359.0,24.210905,0.079518,-79.782396,40.458047,269.336178,0.001775,...,-0.006425,0.008271,-0.119890,-9.812125,4.0,0.0,25,2019-04-07,10:13,R5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257891,279,152.10,1.1,198.0,22.857437,0.095421,-79.782802,40.459018,271.560190,0.021382,...,0.009449,0.444553,-0.274965,-9.796700,10.0,0.0,25-50-100-25,2019-10-24,10:10,R7
257892,279,152.20,1.1,196.0,22.847422,0.095421,-79.782802,40.459018,271.571983,0.021383,...,-0.001755,0.451230,-0.240619,-9.793810,10.0,0.0,25-50-100-25,2019-10-24,10:10,R7
257893,279,152.41,1.2,189.0,22.856186,0.111325,-79.782802,40.459018,271.584533,0.021385,...,0.008545,0.443839,-0.274903,-9.796004,10.0,0.0,25-50-100-25,2019-10-24,10:10,R7
257894,279,152.60,1.1,187.0,22.854933,0.127228,-79.782802,40.459018,271.588050,0.021393,...,-0.001379,0.443880,-0.248434,-9.794703,10.0,0.0,25-50-100-25,2019-10-24,10:10,R7


In [3]:
df = df.sort_values(["flight", "time"]).reset_index(drop=True)
g = df.groupby("flight", group_keys=False)

In [4]:
df["dt"] = g["time"].diff().fillna(0)
 
# battery_current [A] * dt [s] / 3.6 -> mAh consumed in that step
df["capacity_step_mAh"] = (1 / 3.6) * df["dt"] * df["battery_current"]
 
# Energy per step, requested: E[Wh] = V * I * t / 3600
df["energy_step_Wh"] = df["battery_voltage"] * df["battery_current"] * df["dt"] / 3600
 
# wind vector components
df["wind_x"] = df["wind_speed"] * np.sin(np.deg2rad(df["wind_angle"]))
df["wind_y"] = df["wind_speed"] * np.cos(np.deg2rad(df["wind_angle"]))
 
# velocity / acceleration magnitudes
df["velocity_magnitude"] = np.linalg.norm(
    df[["velocity_x", "velocity_y", "velocity_z"]], axis=1
)
df["acceleration_magnitude"] = np.linalg.norm(
    df[["linear_acceleration_x", "linear_acceleration_y", "linear_acceleration_z"]],
    axis=1,
)
 
# position_x = longitude (deg), position_y = latitude (deg), position_z =
# altitude ASL (m). Convert lon/lat deltas to meters using an
# equirectangular approximation (fine for single-flight scale distances),
# then combine with the true-meter altitude delta.
EARTH_R = 6371000.0  # meters
 
lat_rad = np.deg2rad(df["position_y"])
dlat_deg = g["position_y"].diff().fillna(0)
dlon_deg = g["position_x"].diff().fillna(0)
dz = g["position_z"].diff().fillna(0)
 
dlat_m = np.deg2rad(dlat_deg) * EARTH_R
dlon_m = np.deg2rad(dlon_deg) * EARTH_R * np.cos(lat_rad)
 
df["step_dist_m"] = np.sqrt(dlat_m**2 + dlon_m**2 + dz**2)
 
# ---- ground-relative height ("height of drone vs. the land") -----------
# Use each flight's takeoff sample (first row) as the local ground
# reference, since position_z is ASL and ground elevation differs by site.
ground_ref = g["position_z"].transform("first")
df["height_agl"] = df["position_z"] - ground_ref

In [5]:
flight_summary = g.agg(
    payload=("payload", "first"),
    route=("route", "first"),
    altitude_preset=("altitude", "first"),   # categorical, may be multi-stage
    date=("date", "first"),
    time_day=("time_day", "first"),
 
    duration_s=("time", "max"),
    total_distance_m=("step_dist_m", "sum"),          # corrected ground-track range
 
    wind_speed_mean=("wind_speed", "mean"),
    wind_speed_std=("wind_speed", "std"),
    wind_x_mean=("wind_x", "mean"),
    wind_y_mean=("wind_y", "mean"),
 
    speed_mean=("speed", "mean"),
    speed_max=("speed", "max"),                        # "speed limit" reached
 
    velocity_mag_mean=("velocity_magnitude", "mean"),
    velocity_mag_max=("velocity_magnitude", "max"),
 
    acceleration_mag_mean=("acceleration_magnitude", "mean"),
    acceleration_mag_std=("acceleration_magnitude", "std"),
 
    battery_voltage_mean=("battery_voltage", "mean"),
    battery_voltage_min=("battery_voltage", "min"),
 
    max_height_agl=("height_agl", "max"),              # peak height reached, meters
    final_height_agl=("height_agl", "last"),           # ~0 if properly landed
 
    energy_consumed_Wh=("energy_step_Wh", "sum"),       # NEW: requested energy column
    battery_needed_mAh=("capacity_step_mAh", "sum"),    # <-- TARGET (net mAh consumed)
).reset_index()
 
flight_summary["wind_speed_std"] = flight_summary["wind_speed_std"].fillna(0)
flight_summary["acceleration_mag_std"] = flight_summary["acceleration_mag_std"].fillna(0)
 
# QA flag: flights that didn't come back near the ground reference height
flight_summary["landing_offset_flag"] = flight_summary["final_height_agl"].abs() > 5
 

In [6]:
bad = flight_summary.loc[flight_summary["battery_needed_mAh"] < 0, "flight"].tolist()
if bad:
    print(f"[warning] {len(bad)} flight(s) have negative battery_needed_mAh "
          f"(likely sensor noise), flight ids: {bad}")
 
flight_summary.to_parquet(OUT_PUT, index=False, compression= 'zstd')
print(f"Saved {flight_summary.shape[0]} flights x {flight_summary.shape[1]} cols -> {OUT_PUT}")

[warning] 3 flight(s) have negative battery_needed_mAh (likely sensor noise), flight ids: [211, 212, 213]
Saved 209 flights x 25 cols -> /Users/nguyentantai/Desktop/app/ADY201m_Phase02/Drone-Power-predict/data/processed/flight.parquet
